In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
# 1) LayerNorm: 각 example 내부의 Feature Dimension을 대상으로 계산
# 2) BatchNorm: 같은 feature를 여러 example에 걸쳐 계산

layer_norm = nn.LayerNorm(
    normalized_shape=2,
)

batch_norm = nn.LazyBatchNorm1d()

X = torch.tensor(
    [
        [1.0, 2.0],
        [2.0, 3.0],
    ],
    dtype=torch.float32,
)

# BatchNorm이 현재 Batch의 mean, variance를 사용하도록
# Training mode를 유지한다.
layer_norm.train()
batch_norm.train()

with torch.no_grad():
    layer_norm_output = layer_norm(
        X
    )
    batch_norm_output = batch_norm(
        X
    )
    

print(
    "LayerNorm:"
)
print(
    layer_norm_output
)


print(
    "\nBatchNorm:"
)
print(
    batch_norm_output
)

LayerNorm:
tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000]])

BatchNorm:
tensor([[-1.0000, -1.0000],
        [ 1.0000,  1.0000]])


In [3]:
# AddNorm Implementation

class AddNorm(nn.Module):
    
    def __init__(
        self,
        norm_shape: int,
        dropout: float,
    ) -> None:
        super().__init__()
        
        self.dropout = nn.Dropout(
            p=dropout,
        )
        
        self.layer_norm = nn.LayerNorm(
            normalized_shape=norm_shape,
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
        Y: torch.Tensor,
    ) -> torch.Tensor:
        
        if X.shape != Y.shape:
            raise ValueError(
                "X and Y must have the same shape."
            )
            
        return self.layer_norm(
            X + self.dropout(Y)
        )

In [4]:
# AddNorm shape check

add_norm = AddNorm(
    norm_shape=4,
    dropout=0.5,
)

add_norm.eval()

shape = (
    2,
    3,
    4,
)

residual_input = torch.ones(
    shape
)

sublayer_output = torch.ones(
    shape
)

with torch.no_grad():
    output = add_norm(
        residual_input,
        sublayer_output,
    )
    
print(
    "Residual input:",
    tuple(residual_input.shape),
)

print(
    "Sublayer output:",
    tuple(sublayer_output.shape),
)

print(
    "AddNorm output:",
    tuple(output.shape),
)


d2l.check_shape(
    output,
    shape,
)

Residual input: (2, 3, 4)
Sublayer output: (2, 3, 4)
AddNorm output: (2, 3, 4)
